In [0]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "preprod", "prod"], "Environment")

In [0]:
%run /Workspace/Users/lakshmisas96@gmail.com/orders-project/orders-analytics-pipeline/utils/config_loader

In [0]:
import json

config = load_config_from_widget()
print(f"🧪 Running Data Quality Tests - {config['environment'].upper()}")

# Load DQ config
dq_config_path = "/Workspace/Users/lakshmisas96@gmail.com/orders-project/orders-analytics-pipeline/utils/dq_config.json"

with open(dq_config_path, "r") as f:
    dq_config = json.load(f)

print(f"✅ DQ Config loaded!")
print(f"📋 Bronze min rows: {dq_config['bronze']['min_rows']}")
print(f"📋 Bronze max rows: {dq_config['bronze']['max_rows']}")
print(f"📋 Silver min retention: {dq_config['silver']['min_retention_rate']}%")

In [0]:
# In real companies, ALL test results are logged to a table
# So you can track history, build dashboards, set alerts

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {config['catalog']}.{config['schemas']['gold']}.dq_results (
        run_id STRING,
        environment STRING,
        layer STRING,
        table_name STRING,
        test_name STRING,
        status STRING,
        expected STRING,
        actual STRING,
        error_message STRING,
        run_timestamp TIMESTAMP
    )
    USING DELTA
""")

print("✅ DQ Results table ready")

In [0]:
from pyspark.sql.functions import col, countDistinct
from datetime import datetime
import uuid

# Every pipeline run gets a unique ID
# So you can track "run at 9am passed, run at 2pm failed"
run_id = str(uuid.uuid4())[:8]
run_timestamp = datetime.now()
dq_results = []

def log_result(layer, table_name, test_name, status, expected, actual, error_message=""):
    dq_results.append({
        "run_id": run_id,
        "environment": config['environment'],
        "layer": layer,
        "table_name": table_name,
        "test_name": test_name,
        "status": status,
        "expected": str(expected),
        "actual": str(actual),
        "error_message": error_message,
        "run_timestamp": run_timestamp
    })
    icon = "✅" if status == "PASS" else "❌"
    print(f"{icon} [{layer}] {test_name}: {status} (Expected: {expected}, Actual: {actual})")

def run_test(layer, table_name, test_name, condition, expected, actual, error_msg=""):
    if condition:
        log_result(layer, table_name, test_name, "PASS", expected, actual)
    else:
        log_result(layer, table_name, test_name, "FAIL", expected, actual, error_msg)

print(f"🔑 Run ID: {run_id}")
print(f"🕐 Run Timestamp: {run_timestamp}")

In [0]:
print("\n" + "="*50)
print("🥉 BRONZE LAYER TESTS")
print("="*50)

bronze_cfg = dq_config['bronze']
bronze_table = f"{config['catalog']}.{config['schemas']['bronze']}.{bronze_cfg['table']}"
df_bronze = spark.table(bronze_table)
bronze_count = df_bronze.count()

# Test 1: Row count from config
run_test("BRONZE", bronze_table, "row_count_threshold",
    condition = bronze_cfg['min_rows'] <= bronze_count <= bronze_cfg['max_rows'],
    expected = f"between {bronze_cfg['min_rows']}-{bronze_cfg['max_rows']}",
    actual = bronze_count,
    error_msg = "Row count outside expected range!"
)

# Test 2: NULL checks from config
for col_name in bronze_cfg['critical_columns']:
    null_count = df_bronze.filter(col(col_name).isNull()).count()
    run_test("BRONZE", bronze_table, f"no_nulls_{col_name}",
        condition = null_count == 0,
        expected = 0,
        actual = null_count,
        error_msg = f"NULL values found in {col_name}!"
    )

# Test 3: Valid values from config
for column, valid_vals in bronze_cfg['valid_values'].items():
    invalid_count = df_bronze.filter(~col(column).isin(valid_vals)).count()
    run_test("BRONZE", bronze_table, f"valid_{column}_values",
        condition = invalid_count == 0,
        expected = 0,
        actual = invalid_count,
        error_msg = f"Invalid {column} values found!"
    )

In [0]:
print("\n" + "="*50)
print("🥈 SILVER LAYER TESTS")
print("="*50)

silver_cfg = dq_config['silver']
silver_table = f"{config['catalog']}.{config['schemas']['silver']}.{silver_cfg['table']}"
df_silver = spark.table(silver_table)
silver_count = df_silver.count()

# Test 1: Silver <= Bronze
run_test("SILVER", silver_table, "silver_lte_bronze",
    condition = silver_count <= bronze_count,
    expected = f"<= {bronze_count}",
    actual = silver_count,
    error_msg = "Silver has MORE records than Bronze!"
)

# Test 2: Retention rate from config
retention_rate = (silver_count / bronze_count) * 100
run_test("SILVER", silver_table, "record_retention_rate",
    condition = retention_rate >= silver_cfg['min_retention_rate'],
    expected = f">= {silver_cfg['min_retention_rate']}%",
    actual = f"{retention_rate:.2f}%",
    error_msg = "Too many records dropped in Silver!"
)

# Test 3: NULL checks from config
for col_name in silver_cfg['critical_columns']:
    null_count = df_silver.filter(col(col_name).isNull()).count()
    run_test("SILVER", silver_table, f"no_nulls_{col_name}",
        condition = null_count == 0,
        expected = 0,
        actual = null_count,
        error_msg = f"NULL values found in {col_name}!"
    )

# Test 4: Valid values from config
for column, valid_vals in silver_cfg['valid_values'].items():
    invalid_count = df_silver.filter(~col(column).isin(valid_vals)).count()
    run_test("SILVER", silver_table, f"valid_{column}_values",
        condition = invalid_count == 0,
        expected = 0,
        actual = invalid_count,
        error_msg = f"Invalid {column} values found!"
    )

# Test 5: Business logic from config
threshold = silver_cfg['business_logic']['is_high_value_threshold']
wrong_high_value = df_silver.filter(
    ((col("total_amount") >= threshold) & (col("is_high_value") == False)) |
    ((col("total_amount") < threshold) & (col("is_high_value") == True))
).count()
run_test("SILVER", silver_table, "is_high_value_logic_correct",
    condition = wrong_high_value == 0,
    expected = 0,
    actual = wrong_high_value,
    error_msg = f"is_high_value logic broken! Threshold: {threshold}"
)

In [0]:
print("\n" + "="*50)
print("🥇 GOLD LAYER TESTS")
print("="*50)

# ✅ ADD THESE 4 LINES AT TOP
gold_cfg = dq_config['gold']
fact_table = f"{config['catalog']}.{config['schemas']['gold']}.{gold_cfg['fact_table']}"
df_gold = spark.table(fact_table)
gold_count = df_gold.count()

# Test 1: Gold matches Silver
run_test("GOLD", fact_table, "gold_matches_silver",
    condition = gold_count == silver_count,
    expected = silver_count,
    actual = gold_count,
    error_msg = "Gold FactOrders count doesn't match Silver!"
)

# Test 2: DimCustomers has data
dim_customers = spark.table(f"{config['catalog']}.{config['schemas']['gold']}.DimCustomers").count()
run_test("GOLD", "DimCustomers", "dim_customers_has_data",
    condition = dim_customers > 0,
    expected = "> 0",
    actual = dim_customers,
    error_msg = "DimCustomers is empty!"
)

# Test 3: DimProducts has data
dim_products = spark.table(f"{config['catalog']}.{config['schemas']['gold']}.DimProducts").count()
run_test("GOLD", "DimProducts", "dim_products_has_data",
    condition = dim_products > 0,
    expected = "> 0",
    actual = dim_products,
    error_msg = "DimProducts is empty!"
)

# Test 4: No orphan records
orphan_customers = df_gold.join(
    spark.table(f"{config['catalog']}.{config['schemas']['gold']}.DimCustomers"),
    "DimCustomerKey", "left_anti"
).count()
run_test("GOLD", fact_table, "no_orphan_customer_records",
    condition = orphan_customers == 0,
    expected = 0,
    actual = orphan_customers,
    error_msg = "FactOrders has DimCustomerKeys not in DimCustomers!"
)

In [0]:
print("\n" + "="*50)
print("📊 SAVING RESULTS & FINAL SUMMARY")
print("="*50)

# Save all results to Delta table for history tracking
from pyspark.sql import Row
df_results = spark.createDataFrame([Row(**r) for r in dq_results])
df_results.write.format("delta").mode("append").saveAsTable(
    f"{config['catalog']}.{config['schemas']['gold']}.dq_results"
)

# Summary
total_tests = len(dq_results)
passed = len([r for r in dq_results if r['status'] == 'PASS'])
failed = len([r for r in dq_results if r['status'] == 'FAIL'])

print(f"\n📊 DQ SUMMARY - Run ID: {run_id}")
print(f"Environment : {config['environment'].upper()}")
print(f"Total Tests : {total_tests}")
print(f"✅ Passed   : {passed}")
print(f"❌ Failed   : {failed}")
print(f"Pass Rate   : {(passed/total_tests)*100:.1f}%")

# In real companies - FAIL the pipeline if any test fails!
if failed > 0:
    failed_tests = [r['test_name'] for r in dq_results if r['status'] == 'FAIL']
    raise Exception(f"❌ {failed} DATA QUALITY TESTS FAILED: {failed_tests}")
else:
    print(f"\n🎉 ALL {total_tests} TESTS PASSED! Pipeline is healthy!")

In [0]:
print("\n" + "="*50)
print("📊 DQ RESULTS SUMMARY TABLE")
print("="*50)

# Current run results
summary_df = spark.sql(f"""
    SELECT 
        run_id,
        environment,
        layer,
        test_name,
        status,
        expected,
        actual,
        error_message,
        run_timestamp
    FROM {config['catalog']}.{config['schemas']['gold']}.dq_results
    WHERE run_id = '{run_id}'
    ORDER BY layer, test_name
""")

summary_df.display()

print(f"\n📈 HISTORICAL RUNS (Last 10):")

# History of all runs
history_df = spark.sql(f"""
    SELECT 
        run_id,
        environment,
        run_timestamp,
        COUNT(*) as total_tests,
        SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) as passed,
        SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) as failed
    FROM {config['catalog']}.{config['schemas']['gold']}.dq_results
    GROUP BY run_id, environment, run_timestamp
    ORDER BY run_timestamp DESC
    LIMIT 10
""")

history_df.display()

In [0]:
%sql
-- Check what columns FactOrders actually has
DESCRIBE TABLE databricks_catalog_new.dev_gold.FactOrders;

In [0]:
%sql
-- Check what columns DimCustomers has
DESCRIBE TABLE databricks_catalog_new.dev_gold.DimCustomers;

In [0]:
%sql
-- Manually test the join yourself
SELECT COUNT(*) 
FROM databricks_catalog_new.dev_gold.FactOrders f
LEFT ANTI JOIN databricks_catalog_new.dev_gold.DimCustomers d
ON f.DimCustomerKey = d.DimCustomerKey;